# Chapter 05-02 · Simple linear regression, fitted by hand

**Label:** Core  |  **Time:** ~60 minutes  |  **Difficulty:** moderate - the arithmetic is the lesson

**Prerequisites:** 05-01 for the baseline, 03-06 for slopes and units, 03-08 for loss.

**Position in the learning path:** module 05, chapter 2 of 12.

---

## Why this matters

05-01's predictor said the same number for every delivery. This one lets the prediction **depend on
something**: a line instead of a level.

That is the smallest possible step up in complexity, and it is worth taking slowly, because **everything
later is this with more columns**. Multiple regression is this with a vector. Ridge is this with a
penalty. A neural network is this repeated with a squashing function in between. If the arithmetic of one
line is genuinely yours, the rest is bookkeeping.

So this chapter fits the line **by hand** - two sums, a division, a subtraction - and only then checks
against a library. The dataset is nine deliveries chosen so every intermediate number is exact: the means
are whole numbers, the slope is 3.5, and the residuals are halves.

## What you will be able to do

- Fit a least-squares line with a pencil, and say what each sum in the formula is doing
- Read a slope and an intercept in the units of the problem
- Show that the residuals must sum to zero, and why that is not a coincidence
- Connect the slope to the correlation, and R-squared to 05-01's skill score
- Say what "least squares" chose, and what it would have chosen under a different loss

## Warm-up: retrieve, do not reread

1. What constant minimises mean squared error, and what constant minimises mean absolute error?
2. In 03-06, what did a coefficient of 1.8 on `distance_km` mean, in words?
3. What is a skill score?

<br>

*Answers: (1) the mean and the median. (2) one more kilometre is associated with 1.8 more minutes. (3)
`(baseline error - model error) / baseline error`.*

## Nine deliveries, and how far each one travelled

05-01 had delivery times and nothing else, so the best it could do was a constant. Now each delivery comes
with a **distance**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# TINY, SYNTHETIC: nine deliveries. Distances 1 to 9 km, times in minutes.
distance = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
minutes = np.array([16, 20, 25, 25, 30, 31, 36, 42, 45], dtype=float)

deliveries = pd.DataFrame({"distance_km": distance.astype(int), "minutes": minutes.astype(int)})
print(deliveries.to_string(index=False))
print()
print("mean distance %.1f km      mean time %.1f minutes" % (distance.mean(), minutes.mean()))
print("05-01's baseline: predict %.1f minutes for everybody" % minutes.mean())
print("   its mean squared error  %.4f" % ((minutes - minutes.mean()) ** 2).mean())
print("   its mean absolute error %.4f" % np.abs(minutes - minutes.mean()).mean())

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.8))
ax.scatter(distance, minutes, s=90, color="#0072B2", zorder=3, label="the nine deliveries")
ax.axhline(minutes.mean(), color="#999999", linestyle="--", linewidth=2,
           label="05-01's best constant (%.0f min)" % minutes.mean())
for x_value, y_value in zip(distance, minutes):
    ax.plot([x_value, x_value], [minutes.mean(), y_value], color="#cccccc", linewidth=1.4, zorder=1)
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_title("The constant ignores distance. The grey lines are what it gets wrong", fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The grey lines are the constant's errors**, and they are not random - they are short in the middle and
long at both ends, sloping systematically. A predictor that ignores an obvious pattern is leaving
something on the table, and the pattern here is a straight line.

## Fitting it by hand

A line has two numbers: a **slope** (minutes per kilometre) and an **intercept** (minutes at zero
kilometres). Least squares picks the pair that minimises the sum of squared vertical distances, and the
answer has a closed form:

> **slope** = `sum of (x - x̄)(y - ȳ)` divided by `sum of (x - x̄)²`
>
> **intercept** = `ȳ - slope × x̄`

Both sums are worth computing column by column, because each has a meaning.

In [ ]:
mean_distance, mean_minutes = distance.mean(), minutes.mean()

table = pd.DataFrame({
    "distance": distance.astype(int),
    "minutes": minutes.astype(int),
    "x - xbar": distance - mean_distance,
    "y - ybar": minutes - mean_minutes,
})
table["product"] = table["x - xbar"] * table["y - ybar"]
table["(x - xbar) squared"] = table["x - xbar"] ** 2

print(table.to_string(index=False))
print("-" * 62)
print("%-31s sums:  %8.1f  %8.1f" % ("", table["product"].sum(), table["(x - xbar) squared"].sum()))
print()
slope = table["product"].sum() / table["(x - xbar) squared"].sum()
intercept = mean_minutes - slope * mean_distance
print("slope     = %.1f / %.1f = %.4f minutes per km" % (table["product"].sum(),
                                                         table["(x - xbar) squared"].sum(), slope))
print("intercept = %.1f - %.4f x %.1f = %.4f minutes" % (mean_minutes, slope, mean_distance, intercept))

**Slope 3.5 minutes per kilometre. Intercept 12.5 minutes.**

Every number in that table is small enough to check on paper, and the two sums are worth naming:

- **`sum of (x - x̄)(y - ȳ)` = 210.** This is how the two columns move *together*. A delivery that is
  further than average and slower than average contributes a positive product; so does one that is nearer
  and faster. A delivery that breaks the pattern contributes a negative one. **The sum is positive when
  the cloud slopes upward**, and its size grows with both the strength of the relationship and the spread
  of the data.
- **`sum of (x - x̄)²` = 60.** This is how much the distances vary on their own, and it does the job of
  converting the first sum into *minutes per kilometre* rather than a bare number.

The first sum has a picture, and it is the clearest way to see what "moving together" means.

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5.4))
for x_value, y_value in zip(distance, minutes):
    width, height = x_value - mean_distance, y_value - mean_minutes
    positive = width * height >= 0
    ax.add_patch(plt.Rectangle((min(mean_distance, x_value), min(mean_minutes, y_value)),
                               abs(width), abs(height),
                               facecolor="#0072B2" if positive else "#D55E00",
                               alpha=0.22, edgecolor="none"))
ax.scatter(distance, minutes, s=70, color="#000000", zorder=3)
ax.axvline(mean_distance, color="#666666", linewidth=1.4)
ax.axhline(mean_minutes, color="#666666", linewidth=1.4)
ax.text(mean_distance + 0.12, 47, "mean distance", color="#666666", fontsize=9)
ax.text(0.2, mean_minutes + 0.7, "mean minutes", color="#666666", fontsize=9)
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_xlim(0, 10)
ax.set_ylim(13, 49)
ax.set_title("Each delivery draws a rectangle. Blue counts positive, orange negative", fontsize=11.5)
plt.tight_layout()
plt.show()

signed = (distance - mean_distance) * (minutes - mean_minutes)
print("rectangles pulling the slope up   (blue)  : %d, total area %.1f"
      % ((signed > 0).sum(), signed[signed > 0].sum()))
print("rectangles pulling it down        (orange): %d, total area %.1f"
      % ((signed < 0).sum(), -signed[signed < 0].sum() if (signed < 0).any() else 0.0))
print("net                                       : %.1f" % signed.sum())

**Every delivery draws a rectangle** with one corner at the point and the opposite corner at
`(mean distance, mean time)`. Its area is exactly that row's `product` column, and its colour is the sign:
blue when the delivery is on the same side of both means, orange when it breaks the pattern.

`sum of (x - x̄)(y - ȳ)` is therefore **the blue area minus the orange area** - here 210 with no orange at
all, because not one of these nine deliveries is far-and-fast or near-and-slow. That is what a correlation
of 0.99 looks like drawn out.

Now check the fit against a library.

In [ ]:
from sklearn.linear_model import LinearRegression

library = LinearRegression().fit(distance.reshape(-1, 1), minutes)
by_polyfit = np.polyfit(distance, minutes, 1)

print("%-24s %10s %12s" % ("", "slope", "intercept"))
print("%-24s %10.6f %12.6f" % ("by hand", slope, intercept))
print("%-24s %10.6f %12.6f" % ("sklearn LinearRegression", library.coef_[0], library.intercept_))
print("%-24s %10.6f %12.6f" % ("numpy polyfit", by_polyfit[0], by_polyfit[1]))

Identical to six decimal places. **There is no approximation anywhere** - the closed form *is* the answer,
and the library is doing the same arithmetic faster.

### Reading the two numbers

**Slope 3.5 minutes per kilometre.** A delivery one kilometre further away takes three and a half minutes
longer, on average, across these nine deliveries. That is a rate, and it has units - 03-06's point, and
the sentence a stakeholder can act on.

**Intercept 12.5 minutes.** The predicted time for a delivery of zero kilometres. Which is not a delivery
anybody makes, so read it as **the fixed overhead** the line implies: picking up, packing, handing over.
It is a real quantity here because zero is close to the data (distances start at 1 km). **Where zero is far
outside the data, the intercept is an extrapolation and should not be interpreted at all** - a caution that
returns in 05-03.

## What the line does, and what it leaves behind

In [ ]:
predicted = intercept + slope * distance
residual = minutes - predicted

fig, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.8))

grid = np.linspace(0, 10, 2)
left.scatter(distance, minutes, s=90, color="#0072B2", zorder=3)
left.plot(grid, intercept + slope * grid, color="#D55E00", linewidth=2.4,
          label="minutes = %.1f + %.1f x distance" % (intercept, slope))
for x_value, actual, fitted in zip(distance, minutes, predicted):
    left.plot([x_value, x_value], [fitted, actual], color="#009E73", linewidth=2.2, zorder=2)
left.plot([], [], color="#009E73", linewidth=2.2, label="residuals")
left.set_xlabel("distance (km)")
left.set_ylabel("minutes")
left.set_xlim(0, 10)
left.set_title("The fitted line, and what it misses", fontsize=11.5)
left.legend(fontsize=8.5)

right.axhline(0, color="#000000", linewidth=1.2)
right.stem(distance, residual, linefmt="#009E73", markerfmt="o", basefmt=" ")
right.set_xlabel("distance (km)")
right.set_ylabel("residual (minutes)")
right.set_ylim(-3.2, 3.2)
right.set_title("The residuals alone: no slope left, and they sum to zero", fontsize=11.5)

plt.tight_layout()
plt.show()

print("residuals:", residual)
print("they sum to           %.10f" % residual.sum())
print("and, times distance,  %.10f" % (residual * distance).sum())

**Both of those zeros are guaranteed, not lucky.**

They are the two conditions that define the least-squares solution, and each says something you can see in
the picture:

- **The residuals sum to zero.** If they did not, shifting the whole line up or down would reduce the
  squared error - so the best line always passes through the point `(x̄, ȳ)`. **The line goes through the
  middle of the cloud by construction.**
- **The residuals times the distances sum to zero.** If they did not, tilting the line would help. This
  says the leftovers carry **no remaining linear relationship with distance** - the line has taken all of
  it.

That second one is the useful diagnostic. **A residual plot with a visible slope means the fit is wrong**;
a residual plot with visible *curvature* means a line was the wrong shape, which is 05-05's subject.

## Why these formulas, and not some others

The two conditions above are where the formulas come from, and it is worth seeing the loss surface once,
because it is 03-08's bowl with real axes.

**Predict before running:** how many minima does the surface have?

In [ ]:
def squared_error(candidate_intercept, candidate_slope):
    return np.mean((minutes - (candidate_intercept + candidate_slope * distance)) ** 2)


slope_axis = np.linspace(1.5, 5.5, 160)
intercept_axis = np.linspace(2.0, 23.0, 160)
surface = np.array([[squared_error(b, w) for w in slope_axis] for b in intercept_axis])

fig, ax = plt.subplots(figsize=(8, 5.4))
from matplotlib.colors import LogNorm
bands = np.geomspace(surface.min() + 1e-9, surface.max(), 25)
ax.contourf(*np.meshgrid(slope_axis, intercept_axis), surface, levels=bands, norm=LogNorm(),
            cmap="Blues_r")
ax.contour(*np.meshgrid(slope_axis, intercept_axis), surface, levels=bands[::3], colors="white",
           linewidths=0.6)
ax.plot([slope], [intercept], "*", color="#D55E00", markersize=20)
ax.annotate("the answer\n(%.1f, %.1f)" % (slope, intercept), (slope, intercept),
            textcoords="offset points", xytext=(26, 20), color="#D55E00", fontsize=10,
            fontweight="bold")
ax.set_xlabel("candidate slope (minutes per km)")
ax.set_ylabel("candidate intercept (minutes)")
ax.set_title("The squared-error surface has exactly one bottom", fontsize=12)
plt.tight_layout()
plt.show()

**One minimum, and the contours are ellipses.** Squared error in the two parameters is a quadratic bowl,
which is why a formula exists at all: setting both partial derivatives to zero gives two linear equations
in two unknowns, and those solve exactly.

That is the difference between this and almost everything later in the course. **Linear regression under
squared error is one of the few models with a closed-form answer.** Change the loss to absolute error and
the bowl becomes a faceted surface with no formula; add a squashing function and it stops being a bowl at
all. Both then need 03-08's walk downhill.

Note also that the ellipses are **tilted**. Slope and intercept are not independent: raise the slope and
the best intercept falls, because the line must still pass through `(x̄, ȳ)`. That tilt is the reason
centring a feature makes a fit better behaved, and it is 03-08's condition number in a two-parameter
picture.

## Comparing candidate lines

"Least squares" is a claim that this line beats every other. Three near-misses, scored:

In [ ]:
candidates = [("the fitted line", intercept, slope),
              ("slope too shallow", intercept + 5.0, slope - 1.0),
              ("slope too steep", intercept - 5.0, slope + 1.0),
              ("right slope, shifted up", intercept + 3.0, slope)]

fig, ax = plt.subplots(figsize=(8.5, 5))
grid = np.linspace(0, 10, 2)
ax.scatter(distance, minutes, s=90, color="#0072B2", zorder=3)
rows = []
for (label, b, w), colour in zip(candidates, ["#D55E00", "#999999", "#999999", "#999999"]):
    ax.plot(grid, b + w * grid, color=colour, linewidth=2.4 if colour == "#D55E00" else 1.5,
            linestyle="-" if colour == "#D55E00" else "--")
    rows.append({"line": label, "intercept": round(b, 2), "slope": round(w, 2),
                 "sum of squared errors": round(float(((minutes - (b + w * distance)) ** 2).sum()), 3)})
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_xlim(0, 10)
ax.set_title("One solid line and three plausible alternatives", fontsize=11.5)
plt.tight_layout()
plt.show()

print(pd.DataFrame(rows).to_string(index=False))

**17.0 against 77.0, 77.0 and 98.0.** The fitted line is not merely good, it is the single best of all
possible lines under this loss - that is what the closed form guarantees.

Two of the alternatives score identically at 77.0, which is not a coincidence: they are equal and opposite
tilts about the centre of the data, and squared error is symmetric.

Both gaps have closed forms worth knowing, because they explain *why* the surface is a bowl:

- **Tilting** the line about the centre by `d` adds `d² × sum of (x - x̄)²` to the error. Here
  `1 × 60 = 60`, and `17 + 60 = 77` for both tilts.
- **Shifting** it up by `s` adds `n × s²`. Here `9 × 9 = 81`, and `17 + 81 = 98`.

Those two terms are the two axes of the ellipse. **Every departure from the best line costs a quadratic
amount**, which is exactly what makes the minimum unique.

## Did it beat the baseline?

05-01's discipline: a model is worth nothing until it is compared.

In [ ]:
def skill(baseline_error, model_error):
    return (baseline_error - model_error) / baseline_error


baseline_squared = float(((minutes - minutes.mean()) ** 2).mean())
baseline_absolute = float(np.abs(minutes - np.median(minutes)).mean())
model_squared = float((residual ** 2).mean())
model_absolute = float(np.abs(residual).mean())

report = pd.DataFrame([
    {"metric": "mean squared error", "best constant": "the mean (30.0)",
     "baseline": round(baseline_squared, 4), "the line": round(model_squared, 4),
     "skill": "%.2f%%" % (100 * skill(baseline_squared, model_squared))},
    {"metric": "mean absolute error", "best constant": "the median (30.0)",
     "baseline": round(baseline_absolute, 4), "the line": round(model_absolute, 4),
     "skill": "%.2f%%" % (100 * skill(baseline_absolute, model_absolute))},
])
print(report.to_string(index=False))

**97.74% and 85.29%.** A genuine result: the line removes almost all of the squared error the constant
was leaving.

And now a connection worth making explicitly, because it turns a formula people memorise into something
they already understand.

In [ ]:
from sklearn.metrics import r2_score

print("R-squared, from the library : %.6f" % r2_score(minutes, predicted))
print("skill under squared error   : %.6f" % skill(baseline_squared, model_squared))
print()
print("1 - (sum of squared residuals) / (sum of squared deviations from the mean)")
print("  = 1 - %.1f / %.1f = %.6f"
      % ((residual ** 2).sum(), ((minutes - minutes.mean()) ** 2).sum(),
         1 - (residual ** 2).sum() / ((minutes - minutes.mean()) ** 2).sum()))

> **R-squared *is* the skill score under squared error, against the mean baseline.** Same number, same
> definition, different name.

That is worth holding onto, because it demystifies R-squared and it inherits every caution 04-02 attached
to skill scores:

- **R-squared of 0 means "no better than predicting the mean"**, and it can be **negative** - a model can
  be worse than the constant, and on held-out data frequently is.
- **It is always measured against a specific baseline**, namely the mean. It says nothing about whether the
  mean was a *good* baseline. 04-02's per-entity constant beat two models and would make every R-squared in
  that chapter look very different.
- **It is unitless**, which makes it comparable across problems and useless for judging whether an error is
  operationally acceptable. 1.37 minutes of average error is a fact a dispatcher can act on; 0.977 is not.

Report both, and lead with the one in minutes.

In [ ]:
total_variation = float(((minutes - minutes.mean()) ** 2).sum())
left_over = float((residual ** 2).sum())

fig, ax = plt.subplots(figsize=(9.5, 2.7))
ax.barh([0], [total_variation], color="#cccccc", height=0.55)
ax.barh([0], [left_over], color="#D55E00", height=0.55)
ax.text(left_over + (total_variation - left_over) / 2, 0,
        "explained by distance: %.0f  (%.2f%% of the total)"
        % (total_variation - left_over, 100 * (1 - left_over / total_variation)),
        ha="center", va="center", fontsize=10)
ax.annotate("left over: %.0f" % left_over, (left_over, -0.3), textcoords="offset points",
            xytext=(10, -4), color="#D55E00", fontsize=9.5, fontweight="bold")
ax.set_ylim(-0.55, 0.45)
ax.set_yticks([])
ax.set_xlim(0, total_variation * 1.02)
ax.set_xlabel("total squared deviation from the mean (minutes squared)")
ax.set_title("R-squared, drawn: how much of the constant's error the line removed", fontsize=11.5)
plt.tight_layout()
plt.show()

## The slope, the correlation, and standardising

One more identity, because it explains what a slope is made of.

In [ ]:
correlation = float(np.corrcoef(distance, minutes)[0, 1])
spread_ratio = minutes.std(ddof=1) / distance.std(ddof=1)

print("correlation between distance and minutes : %.6f" % correlation)
print("sd(minutes) / sd(distance)               : %.6f" % spread_ratio)
print("their product                            : %.6f" % (correlation * spread_ratio))
print("the fitted slope                         : %.6f" % slope)

> **slope = correlation × (spread of y / spread of x)**

Which separates the two things a slope confuses:

- **The correlation, 0.9886**, is the *strength* of the relationship - unitless, between -1 and 1, and
  unchanged if you switch from kilometres to miles.
- **The spread ratio** carries the *units*, converting a unitless strength into minutes per kilometre.

So a large slope does not mean a strong relationship, and a strong relationship does not mean a large
slope. **Change the units of distance from kilometres to metres and the slope divides by a thousand while
the correlation does not move.** 03-06 made this point about reading coefficients; here it falls out of
the algebra.

It also explains what standardising does: if both columns are standardised, both spreads are 1, and **the
slope simply becomes the correlation.**

In [ ]:
standard_distance = (distance - distance.mean()) / distance.std(ddof=1)
standard_minutes = (minutes - minutes.mean()) / minutes.std(ddof=1)
standard_slope = (((standard_distance - standard_distance.mean())
                   * (standard_minutes - standard_minutes.mean())).sum()
                  / ((standard_distance - standard_distance.mean()) ** 2).sum())

print("slope on the raw columns        : %.6f minutes per km" % slope)
print("slope on standardised columns   : %.6f (unitless)" % standard_slope)
print("the correlation                 : %.6f" % correlation)

## What least squares chose, and what it did not

The line minimises **squared** error. 05-01 showed what that implies, and it applies to lines exactly as
it applied to constants.

In [ ]:
def line_error(b, w, power):
    return float(np.mean(np.abs(minutes - (b + w * distance)) ** power))


search_slope = np.linspace(2.5, 4.5, 401)
search_intercept = np.linspace(6.0, 20.0, 401)

rows = []
for power, label in [(1, "absolute error"), (2, "squared error")]:
    best = min(((line_error(b, w, power), b, w) for b in search_intercept for w in search_slope))
    rows.append({"fitted to": label, "intercept": round(best[1], 3), "slope": round(best[2], 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 5))
grid = np.linspace(0, 10, 2)
ax.scatter(distance, minutes, s=90, color="#0072B2", zorder=3)
for row, colour, style in zip(rows, ["#009E73", "#D55E00"], ["--", "-"]):
    ax.plot(grid, row["intercept"] + row["slope"] * grid, color=colour, linewidth=2.4,
            linestyle=style,
            label="fitted to %s: %.2f + %.2f x" % (row["fitted to"], row["intercept"], row["slope"]))
ax.set_xlabel("distance (km)")
ax.set_ylabel("minutes")
ax.set_xlim(0, 10)
ax.set_title("Two losses, two lines - close here, because nothing is far from the line",
             fontsize=11.5)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

**The two lines are close on this data**, because no delivery is wildly off the pattern - the largest
residual is 2.5 minutes. The distinction that dominated 05-01 is muted here for a specific reason: **there
is no outlier for squared error to chase.**

Put one in and the two lines separate immediately, which is E9. The lesson is the same as before and the
demonstration is now about a line rather than a level: **the loss is a choice, it is usually squared error
because squared error has a formula, and "it has a formula" is a reason of convenience rather than of
correctness.**

## Common misconceptions

**"The intercept is the prediction at zero, so it is always meaningful."**
Only if zero is near the data. Here distances start at 1 km so 12.5 minutes of overhead is a real reading;
predict the salary of someone with zero years of experience from data on 20-to-40-year careers and the
intercept is an extrapolation.

**"A big slope means a strong relationship."**
Slope = correlation × spread ratio. Change kilometres to metres and the slope divides by a thousand while
the relationship is identical.

**"R-squared measures how good the model is."**
It measures skill against **the mean**, under **squared error**. It can be negative, and a high value on a
problem where the mean was already a weak baseline says less than it appears to.

**"Least squares is the right way to fit a line."**
It is the way with a closed form. Under absolute error the best line is different, and neither is more
correct than the loss you actually care about.

**"Residuals summing to zero shows the model is good."**
It is guaranteed by the arithmetic, for every least-squares fit, including hopeless ones. It shows the fit
solved its own equations, nothing more.

**"With one feature there is not much that can go wrong."**
The line can be the wrong shape (05-05), the relationship can be driven by a handful of points (E11), and
the units can make the coefficient meaningless (03-06). One feature is where those are easiest to see, not
where they are absent.

## Exercises

Solutions: `solutions/05_regression/05-02_simple_linear_solutions.ipynb`.

### Quick understanding

**E1.** Write the two formulas for the slope and the intercept, and say in words what each of the two sums
in the slope measures.

**E2.** Why must the fitted line pass through the point `(x̄, ȳ)`?

**E3.** What is R-squared, expressed as a skill score, and what does a negative value mean?

### Hand calculation

**E4.** For `x = 1, 2, 3, 4, 5` and `y = 3, 5, 4, 8, 10`: compute both means, then the two sums, then the
slope and intercept. Show the columns.

**E5.** For your fitted line from E4, compute all five residuals and confirm they sum to zero.

**E6.** Using this chapter's data, compute by hand the sum of squared errors of the line
`minutes = 12.5 + 3.5 × distance` after shifting it up by 2 minutes. Use the shift formula, then check it
directly.

**E7.** The distances are in kilometres and the slope is 3.5 minutes per km. Rewrite the fitted line with
distance in **metres**. Give the new slope and intercept, and say which of the two changed and why.

### Coding

**E8.** Write `fit_line(x, y)` returning the slope and intercept from the closed form, with no library
call. Check it against `np.polyfit` on this chapter's data and on 200 random points.

**E9.** Change the last delivery from 45 to 95 minutes. Refit under squared error and under absolute
error, and plot both lines. How far does each move, and which one would you report?

**E10.** Fit the line the other way round - predict distance from minutes - and plot both lines on the
same axes. Are they the same line? Explain what you see using `slope = r x (sy/sx)`.

**E11.** Compute the fit nine times, each time leaving one delivery out. Plot the nine slopes. Which
delivery matters most, and what does that suggest about a nine-point dataset?

**E12.** Verify the two normal equations numerically for a *random* line and for the fitted line: compute
`sum(residual)` and `sum(residual x distance)` for each. Confirm they are zero only for the fitted one.

### Interpretation

**E13.** A colleague reports "R-squared 0.98, the model is excellent". Give two questions you would ask
before agreeing.

**E14.** The dispatcher wants to promise customers a delivery time from the distance. Using this chapter's
line, say what you would promise for a 6 km delivery and why that is **not** simply the line's prediction.

### Debugging

**E15.** Someone fits a line and gets a slope of 0.29 where you expected about 3.5. Name the most likely
cause and the one-line check.

**E16.** A fitted line has residuals that sum to 40, not 0. Give two explanations.

### Exam and interview reasoning

**E17.** "Derive the least-squares slope." Do it in under two minutes from the idea of minimising squared
error, without quoting the formula from memory. Then answer: "what changes if I use absolute error
instead?"

### Transfer to a different situation

**E18.** You are fitting hospital length-of-stay against patient age, on data from patients aged 45 to 92.
Say what the intercept means, whether you would report it, and what you would do to make the intercept
interpretable.

### Explain it to someone non-technical

**E19.** Explain in under 90 words what "the line of best fit" is fitting, and in what sense it is best.

### Optional challenge

**E20.** Show that the least-squares line is unique by proving the surface is convex: pick 200 random
pairs of (intercept, slope), and for each pair check that the error at the midpoint is **no greater** than
the average of the errors at the two ends. Report how often that holds, and say what it would mean for
gradient descent if it ever failed.

In [ ]:
# Your workspace. In memory: distance, minutes, slope, intercept, predicted, residual,
# mean_distance, mean_minutes, table, skill, squared_error, correlation.

## Mastery check

- [ ] Fit a line from the two sums, on paper, and get the same answer as a library
- [ ] Say what each sum in the slope formula measures
- [ ] Read a slope and an intercept with their units, and know when the intercept is meaningless
- [ ] Explain why the residuals sum to zero and why they are uncorrelated with the feature
- [ ] Recognise R-squared as the skill score under squared error
- [ ] Decompose a slope into correlation and spread ratio
- [ ] Say what least squares chose, and what a different loss would have chosen

## What should now feel instinctive

- Computing the baseline before the line, and the skill after it
- Reading a residual plot for slope and curvature rather than for size alone
- Quoting a coefficient with its units attached
- Distrusting an intercept when zero is outside the data
- Treating "squared error" as a choice that happens to have a formula

## Flashcards

| Front | Back |
|---|---|
| Least-squares slope | `sum((x - x̄)(y - ȳ)) / sum((x - x̄)²)`. Here `210 / 60 = 3.5` |
| Least-squares intercept | `ȳ - slope × x̄`. Here `30 - 3.5 × 5 = 12.5` |
| What the numerator measures | How the two columns move together - positive when the cloud slopes up |
| What the denominator does | Converts it into y-units per x-unit |
| The two normal equations | Residuals sum to zero; residuals times x sum to zero |
| Consequence of the first | The line passes through `(x̄, ȳ)` always |
| Consequence of the second | No linear relationship is left in the residuals |
| Cost of tilting the line by d | `d² × sum((x - x̄)²)` - here 60 per unit of slope |
| Cost of shifting it by s | `n × s²` - here 9 per squared minute |
| R-squared | The skill score under squared error against the mean. Here 0.9774 |
| slope = | `correlation × sd(y) / sd(x)` - strength times a unit conversion |
| On standardised columns | The slope *is* the correlation |

## Next

**05-03 · Multiple linear regression and what a coefficient means.** One feature gave a line; several
give a plane, and `X @ w` from 03-07 becomes the prediction.

The new difficulty is not the arithmetic - it is the interpretation. With one feature, "3.5 minutes per
km" means what it says. With two correlated features, each coefficient means *"holding the other
constant"*, which is a claim about data you may not have, and it is where most misreadings of a regression
come from.